In [1]:
import numpy as np
!pip install casadi
!pip install gymnasium
import scipy.integrate as sci
from scipy.interpolate import interp1d
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import casadi as ca
import gymnasium as gym
from gymnasium import spaces
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
import random, os, pickle, time, copy, warnings, itertools
import h5py
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cpu')   # change to 'cuda' if available

print("✓ All imports OK")
print(f"  numpy  {np.__version__}  |  casadi {ca.__version__}  |  torch {torch.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 MB 7.6 MB/s eta 0:00:00
✓ All imports OK
  numpy  2.0.2  |  casadi 3.7.2  |  torch 2.11.0+cpu


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:


class F18Params:
    # ── Mass & Inertia ────────────────────────────────────────────────────
    mass = 15096.5          # kg
    Ixx  =  31183.6         # kg·m²
    Iyy  = 205127.5
    Izz  = 230432.1
    Ixz  =   4028.0

    # ── Geometry ──────────────────────────────────────────────────────────
    S_ref = 37.16           # m²  wing area
    c_bar =  3.511          # m   mean aero chord
    b     = 11.404          # m   span

    # ── Lift / Drag ───────────────────────────────────────────────────────
    CL_alpha  =  5.0
    CD0       =  0.020
    k         =  0.080      # induced drag factor
    k_alpha2  =  0.300      # nonlinear alpha² drag (FIX from MATLAB)

    # ── Pitching Moment ───────────────────────────────────────────────────
    Cm_alpha   = -0.45
    Cm_q       = -4.50
    Cm_delta_e = -0.50

    # ── Rolling Moment ────────────────────────────────────────────────────
    Cl_beta    = -0.080
    Cl_p       = -0.300
    Cl_r       =  0.050
    Cl_delta_a = -0.150
    Cl_delta_r =  0.050

    # ── Yawing Moment ─────────────────────────────────────────────────────
    Cn_beta    =  0.080
    Cn_p       = -0.030
    Cn_r       = -0.150
    Cn_delta_a = -0.010
    Cn_delta_r = -0.120

    # ── Side Force  ─────────────────
    CY_beta = -0.730
    CY_dr   =  0.200

    # ── Control surface limits ────────────────────────────────────────────
    delta_e_min = np.deg2rad(-24.0);  delta_e_max = np.deg2rad(10.5)
    delta_a_min = np.deg2rad(-25.0);  delta_a_max = np.deg2rad(45.0)
    delta_r_min = np.deg2rad(-30.0);  delta_r_max = np.deg2rad(30.0)
    thrust_min  = 500.0;              thrust_max  = 143200.0

A = F18Params()   # global singleton

print("✓ F-18 HARV parameters loaded")
print(f"  mass={A.mass} kg  |  S={A.S_ref} m²  |  b={A.b} m  |  c̄={A.c_bar} m")
print(f"  T ∈ [{A.thrust_min:.0f}, {A.thrust_max:.0f}] N")
print(f"  δe ∈ [{np.rad2deg(A.delta_e_min):.1f}°, {np.rad2deg(A.delta_e_max):.1f}°]")
print(f"  δa ∈ [{np.rad2deg(A.delta_a_min):.1f}°, {np.rad2deg(A.delta_a_max):.1f}°]")
print(f"  δr ∈ [{np.rad2deg(A.delta_r_min):.1f}°, {np.rad2deg(A.delta_r_max):.1f}°]")


✓ F-18 HARV parameters loaded
  mass=15096.5 kg  |  S=37.16 m²  |  b=11.404 m  |  c̄=3.511 m
  T ∈ [500, 143200] N
  δe ∈ [-24.0°, 10.5°]
  δa ∈ [-25.0°, 45.0°]
  δr ∈ [-30.0°, 30.0°]


In [4]:

# ═══════════════════════════════════════════════════════════════════════════
#  6-DOF EOM — NumPy (for simulation / Gym env)
#  State indexing (0-based):
#    0:u  1:v  2:w  3:p  4:q  5:r  6:q0  7:q1  8:q2  9:q3  10:xN  11:yE  12:zD
#  Control: 0:T  1:de  2:da  3:dr
# ═══════════════════════════════════════════════════════════════════════════

g_acc = 9.81
rho   = 1.225   # sea-level density (constant atmosphere for simplicity)

def quat_to_dcm(q):
    #"\"\"Body→Inertial DCM from quaternion [q0,q1,q2,q3] (scalar-first).\"\"\"
    q0,q1,q2,q3 = q
    return np.array([
        [1-2*(q2**2+q3**2),   2*(q1*q2-q0*q3),   2*(q1*q3+q0*q2)],
        [  2*(q1*q2+q0*q3), 1-2*(q1**2+q3**2),   2*(q2*q3-q0*q1)],
        [  2*(q1*q3-q0*q2),   2*(q2*q3+q0*q1), 1-2*(q1**2+q2**2)]
    ])

def f18_6dof_numpy(t, x, ctrl):
    #"\"\"
    #Continuous 6-DOF F-18 HARV EOM.
    #Matches F18_6DOF_casadi() in MATLAB reference exactly.
    #"\"\"
    u_b,v_b,w_b = x[0],x[1],x[2]
    p,  q,  r   = x[3],x[4],x[5]
    q0, q1, q2, q3 = x[6],x[7],x[8],x[9]

    T, de, da, dr = ctrl
    T  = np.clip(T,  A.thrust_min, A.thrust_max)
    de = np.clip(de, A.delta_e_min, A.delta_e_max)
    da = np.clip(da, A.delta_a_min, A.delta_a_max)
    dr = np.clip(dr, A.delta_r_min, A.delta_r_max)

    # ── Airspeed, alpha, beta ─────────────────────────────────────────────
    V    = np.sqrt(u_b**2 + v_b**2 + w_b**2 + 1e-6)
    qbar = 0.5 * rho * V**2
    alp  = np.arctan2(w_b, u_b)
    bet  = np.arcsin(np.clip(v_b / V, -1, 1))

    # ── Aero coefficients ─────────────────────────────────────────────────
    CL = A.CL_alpha * alp
    CD = A.CD0 + A.k * CL**2 + A.k_alpha2 * alp**2
    CY = A.CY_beta * bet + A.CY_dr * dr

    L_f = qbar * A.S_ref * CL
    D_f = qbar * A.S_ref * CD
    Y_f = qbar * A.S_ref * CY

    # Wind → Body
    ca_, sa_ = np.cos(alp), np.sin(alp)
    cb_, sb_ = np.cos(bet), np.sin(bet)
    Cwb = np.array([
        [ ca_*cb_,  sb_,  sa_*cb_],
        [-ca_*sb_,  cb_, -sa_*sb_],
        [-sa_,      0.,   ca_    ]
    ])
    Fb = Cwb.T @ np.array([T - D_f, Y_f, -L_f])

    # ── Gravity in body frame ─────────────────────────────────────────────
    Rbi = quat_to_dcm([q0,q1,q2,q3])   # body→inertial
    Fg  = A.mass * (Rbi.T @ np.array([0., 0., g_acc]))

    # ── Translational EOM ─────────────────────────────────────────────────
    FX, FY_b, FZ = Fb[0]+Fg[0], Fb[1]+Fg[1], Fb[2]+Fg[2]
    u_dot = FX/A.mass + r*v_b - q*w_b
    v_dot = FY_b/A.mass + p*w_b - r*u_b
    w_dot = FZ/A.mass + q*u_b - p*v_b

    # ── Moments ───────────────────────────────────────────────────────────
    bV = A.b / (2*V)
    cV = A.c_bar / (2*V)

    Cl = (A.Cl_beta*bet + A.Cl_p*bV*p + A.Cl_r*bV*r
          + A.Cl_delta_a*da + A.Cl_delta_r*dr)
    Cm = (A.Cm_alpha*alp + A.Cm_q*cV*q + A.Cm_delta_e*de)
    Cn = (A.Cn_beta*bet + A.Cn_p*bV*p + A.Cn_r*bV*r
          + A.Cn_delta_a*da + A.Cn_delta_r*dr)

    Lm = qbar * A.S_ref * A.b     * Cl
    Mm = qbar * A.S_ref * A.c_bar * Cm
    Nm = qbar * A.S_ref * A.b     * Cn

    I3 = np.array([[A.Ixx, 0, -A.Ixz],
                   [0,     A.Iyy, 0  ],
                   [-A.Ixz,0,  A.Izz ]])
    om  = np.array([p, q, r])
    om_dot = np.linalg.solve(I3, np.array([Lm,Mm,Nm]) - np.cross(om, I3 @ om))

    # ── Quaternion kinematics ─────────────────────────────────────────────
    Om = 0.5 * np.array([
        [ 0, -p, -q, -r],
        [ p,  0,  r, -q],
        [ q, -r,  0,  p],
        [ r,  q, -p,  0]
    ])
    qd = Om @ np.array([q0,q1,q2,q3])

    # ── NED position kinematics ───────────────────────────────────────────
    vi = Rbi @ np.array([u_b, v_b, w_b])

    return np.array([u_dot, v_dot, w_dot,
                     om_dot[0], om_dot[1], om_dot[2],
                     qd[0], qd[1], qd[2], qd[3],
                     vi[0], vi[1], vi[2]])


def simulate_f18(x0, u_traj_fn, t_span, max_step=0.02):
    #"\"\"Simulate using scipy RK45. u_traj_fn(t) → control array.\"\"\"
    def rhs(t, x):
        ctrl = u_traj_fn(t)
        xdot = f18_6dof_numpy(t, x, ctrl)
        # Quaternion normalisation (numerical drift fix)
        xdot[6:10] -= x[6:10] * (np.dot(x[6:10], xdot[6:10]))
        return xdot
    sol = sci.solve_ivp(rhs, t_span, x0, method='RK45',
                        max_step=max_step, dense_output=True)
    return sol

print("✓ F-18 HARV 6-DOF EOM (NumPy) defined")
print("  State dim = 13   [u,v,w, p,q,r, q0,q1,q2,q3, xN,yE,zD]")
print("  Ctrl  dim =  4   [T, δe, δa, δr]")


✓ F-18 HARV 6-DOF EOM (NumPy) defined
  State dim = 13   [u,v,w, p,q,r, q0,q1,q2,q3, xN,yE,zD]
  Ctrl  dim =  4   [T, δe, δa, δr]


In [6]:
# ═══════════════════════════════════════════════════════════════════════════
#  Load Trim State / Control Library from .mat file
#
#  Trim state format:
#  x_trim = [u,v,w, p,q,r, q0,q1,q2,q3, xN,yE,zD]
#
#  Trim control format:
#  u_trim = [T, de, da, dr]
#
#  This replaces the manually generated trim primitives.
# ═══════════════════════════════════════════════════════════════════════════

import numpy as np
from scipy.io import loadmat

# ───────────────────────────────────────────────────────────────────────────
# Load .mat file
# ───────────────────────────────────────────────────────────────────────────
MAT_FILE = '/content/drive/My Drive/trim_states.mat'

mat_data = loadmat(MAT_FILE)

# Extract trim states and controls
raw_trim_states   = mat_data['trim_states']
raw_trim_controls = mat_data['trim_controls']

# Number of trim primitives
N_PRIMS = raw_trim_states.shape[0]

# ───────────────────────────────────────────────────────────────────────────
# Helper functions
# ───────────────────────────────────────────────────────────────────────────

def quat_to_euler(q):
    """
    Quaternion [q0,q1,q2,q3] → Euler angles (phi, theta, psi)
    Scalar-first quaternion convention.
    """
    q0, q1, q2, q3 = q

    # Roll (phi)
    phi = np.arctan2(
        2 * (q0*q1 + q2*q3),
        1 - 2 * (q1**2 + q2**2)
    )

    # Pitch (theta)
    sin_theta = 2 * (q0*q2 - q3*q1)
    sin_theta = np.clip(sin_theta, -1.0, 1.0)
    theta = np.arcsin(sin_theta)

    # Yaw (psi)
    psi = np.arctan2(
        2 * (q0*q3 + q1*q2),
        1 - 2 * (q2**2 + q3**2)
    )

    return phi, theta, psi


def extract_flight_params(state):
    """
    Compute V, alpha, phi, gamma from trim state.
    """

    u, v, w = state[0:3]

    # Airspeed
    V = np.sqrt(u**2 + v**2 + w**2)

    # Angle of attack
    alpha = np.arctan2(w, u)

    # Quaternion
    q = state[6:10]

    # Euler angles
    phi, theta, psi = quat_to_euler(q)

    # Approximate flight-path angle
    gamma = theta - alpha

    return {
        'V': V,
        'alpha_deg': np.rad2deg(alpha),
        'phi_deg': np.rad2deg(phi),
        'gamma_deg': np.rad2deg(gamma)
    }


# ───────────────────────────────────────────────────────────────────────────
# Build Trim Library
# ───────────────────────────────────────────────────────────────────────────

def build_trim_library_from_mat(trim_states, trim_controls):

    prims = []

    for i in range(trim_states.shape[0]):

        # MATLAB cell array → numpy vector
        xs = np.array(trim_states[i, 0]).flatten()

        # Control vector
        uc = np.array(trim_controls[i]).flatten()

        # Extract aerodynamic quantities
        params = extract_flight_params(xs)

        primitive = {
            'label'   : f'TRIM-{i:03d}',
            'state'   : xs,
            'control' : uc,

            'V'       : params['V'],
            'alpha'   : params['alpha_deg'],
            'phi'     : params['phi_deg'],
            'gamma'   : params['gamma_deg']
        }

        prims.append(primitive)

    return prims


# ───────────────────────────────────────────────────────────────────────────
# Create Library
# ───────────────────────────────────────────────────────────────────────────

TRIM_LIBRARY = build_trim_library_from_mat(
    raw_trim_states,
    raw_trim_controls
)

# ───────────────────────────────────────────────────────────────────────────
# Summary
# ───────────────────────────────────────────────────────────────────────────

print(f"\n✓ Loaded Trim Library: {len(TRIM_LIBRARY)} primitives\n")

print(f"{'Label':<12} {'V(m/s)':>10} {'α(deg)':>10} "
      f"{'φ(deg)':>10} {'γ(deg)':>10}")

print("-" * 60)

for p in TRIM_LIBRARY[:5]:

    print(f"{p['label']:<12} "
          f"{p['V']:>10.2f} "
          f"{p['alpha']:>10.2f} "
          f"{p['phi']:>10.2f} "
          f"{p['gamma']:>10.2f}")

# Example access:
#
# TRIM_LIBRARY[i]['state']
# TRIM_LIBRARY[i]['control']
#
# Example:
#
# x_trim = TRIM_LIBRARY[0]['state']
# u_trim = TRIM_LIBRARY[0]['control']
#



✓ Loaded Trim Library: 275 primitives

Label            V(m/s)     α(deg)     φ(deg)     γ(deg)
------------------------------------------------------------
TRIM-000         120.00       5.18       0.00       0.00
TRIM-001         130.00       4.41       0.00      -0.00
TRIM-002         140.00       3.80       0.00      -0.00
TRIM-003         150.00       3.31       0.00       0.00
TRIM-004         160.00       2.91       0.00       0.00


In [7]:

V_THRESH = 80.0
A_THRESH = np.deg2rad(15.0)
B_THRESH = np.deg2rad(10.0)
P_THRESH = np.deg2rad(60.0)

def angdiff(a, b):
    #\"\"\"Wrap-safe angle difference ∈ (-π, π].\"\"\"
    d = (a - b + np.pi) % (2*np.pi) - np.pi
    return d

def feasibility_filter(lib):
    #\"\"\"Return list of (i,j) candidate pairs passing the pre-filter.\"\"\"
    candidates = []
    for i in range(len(lib)):
        for j in range(len(lib)):
            if i == j: continue
            pi, pj = lib[i], lib[j]
            dV   = abs(pi['V']     - pj['V'])
            dA   = abs(np.deg2rad(pi['alpha']) - np.deg2rad(pj['alpha']))
            dPhi = abs(angdiff(np.deg2rad(pi['phi']), np.deg2rad(pj['phi'])))
            if dV <= V_THRESH and dA <= A_THRESH and dPhi <= P_THRESH:
                candidates.append((i, j))
    return candidates

CANDIDATES = feasibility_filter(TRIM_LIBRARY)
total = N_PRIMS * (N_PRIMS - 1)
print(f"✓ Feasibility filter: {len(CANDIDATES)} / {total} pairs pass "
      f"({100*len(CANDIDATES)/total:.1f}%)")


✓ Feasibility filter: 32172 / 75350 pairs pass (42.7%)


In [8]:

# ═══════════════════════════════════════════════════════════════════════════
#  Adaptive Tf (mirrors compute_adaptive_Tf in MATLAB)
# ═══════════════════════════════════════════════════════════════════════════
Tf_BASE = 5.0;  Tf_MAX = 25.0

def adaptive_Tf(pi, pj):
    dV   = abs(pi['V'] - pj['V']) / V_THRESH
    dA   = abs(np.deg2rad(pi['alpha'] - pj['alpha'])) / A_THRESH
    dPhi = abs(angdiff(np.deg2rad(pi['phi']), np.deg2rad(pj['phi']))) / P_THRESH
    dGam = abs(np.deg2rad(pi['gamma'] - pj['gamma'])) / np.deg2rad(20.0)
    diff = dV + dA + dPhi + dGam
    Tf   = Tf_BASE + (Tf_MAX - Tf_BASE) * min(diff / 2.0, 1.0)
    return round(Tf) + 2


# ═══════════════════════════════════════════════════════════════════════════
#  CasADi OCP Solver — direct multiple-shooting + IPOPT
# ═══════════════════════════════════════════════════════════════════════════
def solve_ocp(x0_np, xf_np, u0_np, uf_np, Tf, N=80):
    nx, nu = 13, 4
    dt = Tf / N

    # ── Align quaternion signs (shortest path) ────────────────────────────
    if np.dot(x0_np[6:10], xf_np[6:10]) < 0:
        xf_np = xf_np.copy(); xf_np[6:10] *= -1

    # ── Symbolic variables ────────────────────────────────────────────────
    xs = ca.SX.sym('x', nx)
    us = ca.SX.sym('u', nu)

    # ── CasADi 6-DOF EOM (mirrors F18_6DOF_casadi in MATLAB) ─────────────
    def f18_sym(x, u):
        u_b,v_b,w_b = x[0],x[1],x[2]
        p_, q_, r_  = x[3],x[4],x[5]
        q0,q1,q2,q3 = x[6],x[7],x[8],x[9]

        T_,de_,da_,dr_ = u[0],u[1],u[2],u[3]

        V_   = ca.sqrt(u_b**2 + v_b**2 + w_b**2 + 1e-6)
        qb_  = 0.5 * rho * V_**2
        alp_ = ca.atan2(w_b, u_b)
        bet_ = ca.asin(v_b/V_)

        CL_ = A.CL_alpha * alp_
        CD_ = A.CD0 + A.k*CL_**2 + A.k_alpha2*alp_**2
        CY_ = A.CY_beta*bet_ + A.CY_dr*dr_

        L_f = qb_*A.S_ref*CL_; D_f = qb_*A.S_ref*CD_; Y_f = qb_*A.S_ref*CY_

        ca_a = ca.cos(alp_); sa_a = ca.sin(alp_)
        cb_b = ca.cos(bet_); sb_b = ca.sin(bet_)

        # Wind→Body rotation
        # Exact Cwb.T @ [T-D; Y; -L]
        FX_b =  (T_-D_f)*ca_a*cb_b - Y_f*ca_a*sb_b + L_f*sa_a
        FY_b =  (T_-D_f)*sb_b      + Y_f*cb_b
        FZ_b =  (T_-D_f)*sa_a*cb_b - Y_f*sa_a*sb_b - L_f*ca_a

        # Gravity
        Rbi11 = 1-2*(q2**2+q3**2); Rbi12 = 2*(q1*q2-q0*q3); Rbi13 = 2*(q1*q3+q0*q2)
        Rbi21 = 2*(q1*q2+q0*q3);   Rbi22 = 1-2*(q1**2+q3**2); Rbi23 = 2*(q2*q3-q0*q1)
        Rbi31 = 2*(q1*q3-q0*q2);   Rbi32 = 2*(q2*q3+q0*q1);   Rbi33 = 1-2*(q1**2+q2**2)

        # Rbi.T @ [0,0,g] means 3rd row of Rbi.T = 3rd column of Rbi
        Fg1 = A.mass * g_acc * Rbi31
        Fg2 = A.mass * g_acc * Rbi32
        Fg3 = A.mass * g_acc * Rbi33

        u_dot = (FX_b + Fg1)/A.mass + r_*v_b - q_*w_b
        v_dot = (FY_b + Fg2)/A.mass + p_*w_b - r_*u_b
        w_dot = (FZ_b + Fg3)/A.mass + q_*u_b - p_*v_b

        bV = A.b    / (2*V_)
        cV = A.c_bar/ (2*V_)
        Cl_ = A.Cl_beta*bet_ + A.Cl_p*bV*p_ + A.Cl_r*bV*r_ + A.Cl_delta_a*da_ + A.Cl_delta_r*dr_
        Cm_ = A.Cm_alpha*alp_ + A.Cm_q*cV*q_ + A.Cm_delta_e*de_
        Cn_ = A.Cn_beta*bet_ + A.Cn_p*bV*p_ + A.Cn_r*bV*r_ + A.Cn_delta_a*da_ + A.Cn_delta_r*dr_

        Lm_ = qb_*A.S_ref*A.b    *Cl_
        Mm_ = qb_*A.S_ref*A.c_bar*Cm_
        Nm_ = qb_*A.S_ref*A.b    *Cn_

        I3 = ca.vertcat(
            ca.horzcat(A.Ixx, 0, -A.Ixz),
            ca.horzcat(0,     A.Iyy, 0),
            ca.horzcat(-A.Ixz, 0,  A.Izz)
        )
        om_ = ca.vertcat(p_, q_, r_)
        moments = ca.vertcat(Lm_, Mm_, Nm_)

        # Euler's equations using CasADi analytical solve
        I3_om = ca.mtimes(I3, om_)
        cross_term = ca.vertcat(
            om_[1]*I3_om[2] - om_[2]*I3_om[1],
            om_[2]*I3_om[0] - om_[0]*I3_om[2],
            om_[0]*I3_om[1] - om_[1]*I3_om[0]
        )
        om_dot = ca.solve(I3, moments - cross_term)
        p_dot = om_dot[0]
        q_dot = om_dot[1]
        r_dot = om_dot[2]

        # Quaternion kinematics
        qd0 = 0.5*(-p_*q1 - q_*q2 - r_*q3)
        qd1 = 0.5*( p_*q0 + r_*q2 - q_*q3)
        qd2 = 0.5*( q_*q0 - r_*q1 + p_*q3)
        qd3 = 0.5*( r_*q0 + q_*q1 - p_*q2)

        # NED kinematics: vi = Rbi @ [u,v,w]
        vN = Rbi11*u_b + Rbi12*v_b + Rbi13*w_b
        vE = Rbi21*u_b + Rbi22*v_b + Rbi23*w_b
        vD = Rbi31*u_b + Rbi32*v_b + Rbi33*w_b

        return ca.vertcat(u_dot, v_dot, w_dot,
                          p_dot, q_dot, r_dot,
                          qd0,qd1,qd2,qd3,
                          vN, vE, vD)

    f_sym = ca.Function('f', [xs,us], [f18_sym(xs,us)])

    # ── RK4 integrator ────────────────────────────────────────────────────
    def rk4_step(xk, uk, h):
        k1 = f_sym(xk, uk)
        k2 = f_sym(xk + h/2*k1, uk)
        k3 = f_sym(xk + h/2*k2, uk)
        k4 = f_sym(xk + h*k3,   uk)
        return xk + h/6*(k1 + 2*k2 + 2*k3 + k4)

    # ── Warm-start (linear interpolation, SLERP for quaternion) ──────────
    def slerp(q0, q1, t):
        dot = np.clip(np.dot(q0, q1), -1, 1)
        if dot < 0: q1 = -q1; dot = -dot
        th = np.arccos(dot)
        if abs(np.sin(th)) < 1e-8:
            return ((1-t)*q0 + t*q1)
        return (np.sin((1-t)*th)*q0 + np.sin(t*th)*q1) / np.sin(th)

    X_ws = np.zeros((N+1, nx))
    for k in range(N+1):
        tau = k / N
        X_ws[k, :] = (1-tau)*x0_np + tau*xf_np
        X_ws[k, 1] = 0.0  # sideslip guess = 0
        X_ws[k, 6:10] = slerp(x0_np[6:10]/np.linalg.norm(x0_np[6:10]),
                               xf_np[6:10]/np.linalg.norm(xf_np[6:10]), tau)
        # Position extrapolation from initial velocity
        X_ws[k, 10] = x0_np[10] + tau * Tf * x0_np[0]
        X_ws[k, 11] = x0_np[11] + tau * Tf * x0_np[1]
        X_ws[k, 12] = x0_np[12] + tau * Tf * x0_np[2]

    U_ws = np.zeros((N, nu))
    for k in range(N):
        tau = k / N
        U_ws[k] = (1-tau)*u0_np + tau*uf_np

    # ── NLP decision variables ────────────────────────────────────────────
    opti = ca.Opti()
    X    = opti.variable(N+1, nx)
    U    = opti.variable(N,   nu)

    opti.set_initial(X, X_ws)
    opti.set_initial(U, U_ws)

    # ── Dynamics constraints (RK4 collocation) ────────────────────────────
    for k in range(N):
        xk_next = rk4_step(X[k,:].T, U[k,:].T, dt)
        opti.subject_to(X[k+1,:].T == xk_next)

    # ── Boundary conditions ───────────────────────────────────────────────
    opti.subject_to(X[-1, :10] == ca.reshape(xf_np[:10], 1, 10))

    # Terminal tolerances (physical units, from MATLAB tol_term)
    tol_term = np.array([2.0, 1.0, 2.0, 0.05, 0.05, 0.05, 0.02, 0.02, 0.02, 0.02])
    opti.subject_to(opti.bounded(xf_np[:10] - tol_term, X[N, :10].T, xf_np[:10] + tol_term))

    # ── Path constraints ──────────────────────────────────────────────────
    for k in range(N+1):
        # Quaternion exact unit-norm
        q_n = ca.sumsqr(X[k, 6:10])
        opti.subject_to(q_n == 1.0)
        # Sideslip ±10°
        V_k   = ca.sqrt(ca.sumsqr(X[k,:3]) + 1e-6)
        beta_k = X[k,1] / V_k
        opti.subject_to(opti.bounded(-np.deg2rad(10), beta_k, np.deg2rad(10)))

    # ── Control bounds (from MATLAB) ──────────────────────────────────────
    opti.subject_to(opti.bounded(500.0,             U[:,0], 142000.0))
    opti.subject_to(opti.bounded(-np.deg2rad(25.0), U[:,1], np.deg2rad(25.0)))
    opti.subject_to(opti.bounded(-np.deg2rad(21.0), U[:,2], np.deg2rad(21.0)))
    opti.subject_to(opti.bounded(-np.deg2rad(30.0), U[:,3], np.deg2rad(30.0)))

    # Lock endpoints
    opti.subject_to(U[0,  :].T == u0_np)
    opti.subject_to(U[N-1,:].T == uf_np)

    # ── Objective ─────────────────────────────────────────────────────────
    # Exact R weights from MATLAB
    R_diag = ca.vertcat(1/142000.0**2,
                        1/np.deg2rad(25.0)**2,
                        1/np.deg2rad(21.0)**2,
                        1/np.deg2rad(30.0)**2)

    J = 0
    for k in range(N):
        uk = U[k,:].T
        J = J + dt * ca.dot(uk * R_diag, uk)

    opti.minimize(J)

    # ── IPOPT settings ────────────────────────────────────────────────────
    opti.solver('ipopt', {
        'ipopt.max_iter':          2000,
        'ipopt.tol':               1e-5,
        'ipopt.acceptable_tol':    1e-4,
        'ipopt.acceptable_iter':   8,
        'ipopt.print_level':       0,
        'ipopt.acceptable_obj_change_tol': 1e-4,
        'ipopt.sb':                'yes',
        'print_time':              False
    })

    try:
        sol     = opti.solve()
        X_opt   = sol.value(X)
        U_opt   = sol.value(U)
        cost    = sol.value(opti.f)
        status  = 'Solve_Succeeded'
        feasible = True
    except Exception as e:
        # Try to extract debug solution
        try:
            X_opt  = opti.debug.value(X)
            U_opt  = opti.debug.value(U)
        except:
            X_opt  = X_ws
            U_opt  = U_ws
        cost    = float('nan')
        status  = str(e)[:60]
        feasible = False

    T_vec = np.linspace(0, Tf, N+1)
    return dict(feasible=feasible, X=X_opt, U=U_opt, T=T_vec,
                Tf=Tf, cost=cost, status=status)


print("✓ OCP solver matched to MATLAB reference")
print("  Objective: minimise control effort ONLY")
print("  Constraints: RK4 dynamics, quaternion exact norm, bounded terminal states")



✓ OCP solver matched to MATLAB reference
  Objective: minimise control effort ONLY
  Constraints: RK4 dynamics, quaternion exact norm, bounded terminal states


In [9]:

# ═══════════════════════════════════════════════════════════════════════════
#  Build Transition Library (offline)
#  Solves OCP for every feasible (i,j) pair.
#  Results stored in: transition_library[i][j]
# ═══════════════════════════════════════════════════════════════════════════

LIBRARY_FILE = 'transition_library.pkl'
FORCE_REBUILD = False    # Set True to re-solve even if file exists

def build_transition_library(trim_lib, candidates, max_pairs=None):
    #\"\"\"
    #Solve OCP for each (i,j) candidate pair.
    #max_pairs: int or None — limit for quick testing (None = all pairs).
    #\"\"\"
    N_p = len(trim_lib)
    # Initialise with empty entries
    lib = [[dict(feasible=False, X=None, U=None, T=None,
                 Tf=None, cost=np.nan, status='not_attempted')
            for _ in range(N_p)] for _ in range(N_p)]

    cands = candidates if max_pairs is None else candidates[:max_pairs]
    n_ok = 0
    t0   = time.time()

    print(f"{'Pair':<10} {'Tf':>5} {'Status':<35} {'Cost':>12}")
    print("─"*65)

    for idx, (i, j) in enumerate(cands):
        pi, pj = trim_lib[i], trim_lib[j]
        Tf_ij  = adaptive_Tf(pi, pj)

        result = solve_ocp(pi['state'].copy(), pj['state'].copy(),
                   pi['control'].copy(), pj['control'].copy(), Tf_ij)
        lib[i][j] = result
        flag = '✓' if result['feasible'] else '✗'
        cost_str = f"{result['cost']:.4g}" if np.isfinite(result['cost']) else 'NaN'
        print(f"[{flag}] {pi['label']}→{pj['label']:<8} "
              f"Tf={Tf_ij:4.0f}s  {result['status'][:35]:<35}  J={cost_str}")
        if result['feasible']: n_ok += 1

        # Autosave every 20 pairs
        if (idx+1) % 20 == 0:
            with open(LIBRARY_FILE, 'wb') as f:
                pickle.dump(lib, f)
            elapsed = (time.time()-t0)/60
            print(f"  ── autosave ({idx+1}/{len(cands)}, {elapsed:.1f} min) ──")

    print(f"\\n{'─'*65}")
    print(f"DONE: {n_ok}/{len(cands)} converged ({100*n_ok/max(len(cands),1):.1f}%)")
    print(f"Elapsed: {(time.time()-t0)/60:.2f} min")
    return lib


if os.path.exists(LIBRARY_FILE) and not FORCE_REBUILD:
    with open(LIBRARY_FILE, 'rb') as f:
        TRANSITION_LIBRARY = pickle.load(f)
    print(f"✓ Loaded existing transition library from '{LIBRARY_FILE}'")
    n_ok = sum(TRANSITION_LIBRARY[i][j]['feasible']
               for i,j in CANDIDATES)
    print(f"  {n_ok}/{len(CANDIDATES)} feasible transitions")
else:
    print("Building transition library — this may take several minutes...")
    print("(Set max_pairs=N for a quick test with fewer pairs)\\n")
    # Use max_pairs=None for full library; set to e.g. 30 for quick demo
    TRANSITION_LIBRARY = build_transition_library(TRIM_LIBRARY, CANDIDATES,
                                                  max_pairs=5)
    with open(LIBRARY_FILE, 'wb') as f:
        pickle.dump(TRANSITION_LIBRARY, f)
    print(f"\\n✓ Library saved to '{LIBRARY_FILE}'")


Building transition library — this may take several minutes...
(Set max_pairs=N for a quick test with fewer pairs)\n
Pair          Tf Status                                      Cost
─────────────────────────────────────────────────────────────────
[✓] TRIM-000→TRIM-001 Tf=   9s  Solve_Succeeded                      J=0.102
[✓] TRIM-000→TRIM-002 Tf=  10s  Solve_Succeeded                      J=0.1438
[✓] TRIM-000→TRIM-003 Tf=  12s  Solve_Succeeded                      J=0.3166
[✗] TRIM-000→TRIM-004 Tf=  14s  Error in Opti::solve [OptiNode] at   J=NaN
[✓] TRIM-000→TRIM-005 Tf=  15s  Solve_Succeeded                      J=1.331
\n─────────────────────────────────────────────────────────────────
DONE: 4/5 converged (80.0%)
Elapsed: 7.87 min
\n✓ Library saved to 'transition_library.pkl'


In [10]:

def plot_ocp_trajectory(result, label='OCP Trajectory'):
    #\"\"\"Plot position (3D), velocity, body rates, and controls.\"\"\"
    if not result['feasible']:
        print("Trajectory not feasible — skipping plot."); return
    X, U, T = result['X'], result['U'], result['T']
    Tu = T[:-1]

    fig = plt.figure(figsize=(18, 10))
    fig.suptitle(label, fontsize=14, fontweight='bold')

    # 3-D position
    ax = fig.add_subplot(2, 4, 1, projection='3d')
    ax.plot(X[:,10], X[:,11], -X[:,12], 'b-', lw=2)
    ax.scatter(*[X[0,10]], *[X[0,11]], *[-X[0,12]], c='g', s=80, zorder=5, label='Start')
    ax.scatter(*[X[-1,10]], *[X[-1,11]], *[-X[-1,12]], c='r', s=80, zorder=5, label='End')
    ax.set_xlabel('North [m]'); ax.set_ylabel('East [m]'); ax.set_zlabel('Alt [m]')
    ax.set_title('3-D Path'); ax.legend(fontsize=7)

    # Airspeed
    ax2 = fig.add_subplot(2, 4, 2)
    V_arr = np.sqrt(X[:,0]**2 + X[:,1]**2 + X[:,2]**2)
    ax2.plot(T, V_arr, 'b-', lw=2)
    ax2.set_title('Airspeed'); ax2.set_xlabel('t [s]'); ax2.set_ylabel('V [m/s]')
    ax2.grid(True, alpha=0.3)

    # Alpha / Beta
    ax3 = fig.add_subplot(2, 4, 3)
    V_s  = np.sqrt(X[:,0]**2+X[:,1]**2+X[:,2]**2)+1e-6
    alpha_arr = np.degrees(np.arctan2(X[:,2], X[:,0]))
    beta_arr  = np.degrees(np.arcsin(np.clip(X[:,1]/V_s, -1, 1)))
    ax3.plot(T, alpha_arr, 'r-', label='α', lw=2)
    ax3.plot(T, beta_arr,  'b-', label='β', lw=2)
    ax3.axhline(0, color='k', lw=0.5)
    ax3.set_title('α & β'); ax3.set_xlabel('t [s]'); ax3.set_ylabel('[deg]')
    ax3.legend(); ax3.grid(True, alpha=0.3)

    # Body rates (aggressiveness indicator)
    ax4 = fig.add_subplot(2, 4, 4)
    ax4.plot(T, np.degrees(X[:,3]), 'r-', label='p', lw=2)
    ax4.plot(T, np.degrees(X[:,4]), 'g-', label='q', lw=2)
    ax4.plot(T, np.degrees(X[:,5]), 'b-', label='r', lw=2)
    ax4.axhline(0, color='k', lw=0.5)
    ax4.set_title('Body Rates (Aggressiveness)'); ax4.set_xlabel('t [s]')
    ax4.set_ylabel('[deg/s]'); ax4.legend(); ax4.grid(True, alpha=0.3)

    # Thrust
    ax5 = fig.add_subplot(2, 4, 5)
    ax5.plot(Tu, U[:,0]/1000, 'k-', lw=2)
    ax5.set_title('Thrust'); ax5.set_xlabel('t [s]'); ax5.set_ylabel('[kN]')
    ax5.grid(True, alpha=0.3)

    # Control surfaces
    ax6 = fig.add_subplot(2, 4, 6)
    ax6.plot(Tu, np.degrees(U[:,1]), 'r-', label='δe', lw=2)
    ax6.plot(Tu, np.degrees(U[:,2]), 'g-', label='δa', lw=2)
    ax6.plot(Tu, np.degrees(U[:,3]), 'b-', label='δr', lw=2)
    ax6.axhline(0, color='k', lw=0.5)
    ax6.set_title('Control Surfaces'); ax6.set_xlabel('t [s]')
    ax6.set_ylabel('[deg]'); ax6.legend(); ax6.grid(True, alpha=0.3)

    # Altitude
    ax7 = fig.add_subplot(2, 4, 7)
    ax7.plot(T, -X[:,12], 'c-', lw=2)
    ax7.set_title('Altitude'); ax7.set_xlabel('t [s]'); ax7.set_ylabel('[m]')
    ax7.grid(True, alpha=0.3)

    # Quaternion norm (should stay ≈ 1)
    ax8 = fig.add_subplot(2, 4, 8)
    q_norm = np.linalg.norm(X[:,6:10], axis=1)
    ax8.plot(T, q_norm, 'm-', lw=2)
    ax8.axhline(1.0, color='k', lw=0.5, ls='--')
    ax8.set_title('||q|| (should ≡ 1)'); ax8.set_xlabel('t [s]')
    ax8.set_ylabel('norm'); ax8.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{label.replace(" ","_")}.png', dpi=120, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {label.replace(' ','_')}.png")


# ── Plot first feasible pair ──────────────────────────────────────────────────
for (i, j) in CANDIDATES:
    res = TRANSITION_LIBRARY[i][j]
    if res['feasible']:
        lbl = f"OCP: {TRIM_LIBRARY[i]['label']} → {TRIM_LIBRARY[j]['label']}"
        plot_ocp_trajectory(res, lbl)
        print(f"✓ Plotted: {lbl}")
        break


  Saved: OCP:_TRIM-000_→_TRIM-001.png
✓ Plotted: OCP: TRIM-000 → TRIM-001


In [11]:

# ═══════════════════════════════════════════════════════════════════════════
#  F-18 HARV Gymnasium Environment
#  Goal-conditioned: obs = [normalised_state | normalised_goal]
# ═══════════════════════════════════════════════════════════════════════════

# Normalisation ranges (for [-1,1] scaling)
STATE_SCALE = np.array([
    300., 50., 50.,          # u, v, w  [m/s]
    1.0,  1.0, 1.0,          # p, q, r  [rad/s]
    1.0,  1.0, 1.0, 1.0,    # q0,q1,q2,q3
    5000., 5000., 5000.      # xN,yE,zD [m]
])

CTRL_SCALE = np.array([A.thrust_max, A.delta_e_max, A.delta_a_max, A.delta_r_max])
CTRL_OFFSET= np.array([0., 0., 0., 0.])   # controls centred at 0 except thrust


class F18EvasionEnv(gym.Env):
    #\"\"\"
    #Goal-conditioned F-18 HARV evasion environment.

    #Observation : [x_norm | x_goal_norm]  ∈ R^26
    #Action      : [T, de, da, dr] normalised to [-1,1]
    #Reward      : aggressive evasion shaping
    #Episode ends: terminal state reached OR timeout OR unsafe
    #\"\"\"
    metadata = {}

    def __init__(self,
                 trim_library,
                 transition_library,
                 candidates,
                 dt=0.05,
                 max_t=30.0,
                 use_library_goal=True,
                 w_goal=1.0,
                 w_agg=0.5,
                 w_time=0.05,
                 w_ctrl=0.01,
                 r_terminal=200.0):

        super().__init__()
        self.trim_lib   = trim_library
        self.trans_lib  = transition_library
        self.candidates = candidates
        self.dt         = dt
        self.max_t      = max_t
        self.use_lib    = use_library_goal

        # Reward weights
        self.w_goal     = w_goal
        self.w_agg      = w_agg
        self.w_time     = w_time
        self.w_ctrl     = w_ctrl
        self.r_terminal = r_terminal

        # Spaces
        obs_dim = 13 * 2   # [state | goal]
        self.observation_space = spaces.Box(-5., 5., (obs_dim,), dtype=np.float32)
        self.action_space      = spaces.Box(-1., 1., (4,),      dtype=np.float32)

        # State
        self.x      = None
        self.x_goal = None
        self.t      = 0.0
        self.i_src  = None
        self.j_tgt  = None
        self._ref_traj = None   # optional OCP reference for reward shaping

    # ── Normalisation helpers ──────────────────────────────────────────────
    @staticmethod
    def normalise(x):
        return np.clip(x / STATE_SCALE, -5., 5.).astype(np.float32)

    def _get_obs(self):
        return np.concatenate([self.normalise(self.x),
                               self.normalise(self.x_goal)]).astype(np.float32)

    def _denorm_action(self, a):
        #\"\"\"Map action [-1,1]^4 → physical controls.\"\"\"
        T  = (a[0]+1)/2 * (A.thrust_max - A.thrust_min) + A.thrust_min
        de = a[1] * A.delta_e_max
        da = a[2] * A.delta_a_max
        dr = a[3] * A.delta_r_max
        return np.array([T, de, da, dr])

    # ── Reset ─────────────────────────────────────────────────────────────
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        # Sample a random feasible (i,j) pair
        pair = random.choice(self.candidates)
        self.i_src, self.j_tgt = pair

        pi = self.trim_lib[self.i_src]
        pj = self.trim_lib[self.j_tgt]

        # Small noise on initial state (robustness)
        noise = np.zeros(13)
        noise[:3]  = np.random.randn(3) * 2.0    # velocity ±2 m/s
        noise[3:6] = np.random.randn(3) * 0.01   # rate ±0.01 rad/s

        self.x      = pi['state'].copy() + noise
        # Renormalise quaternion
        self.x[6:10] /= np.linalg.norm(self.x[6:10])
        self.x_goal  = pj['state'].copy()
        self.t       = 0.0

        # Store reference trajectory if available (for potential shaping)
        res = self.trans_lib[self.i_src][self.j_tgt]
        self._ref_traj = res if res['feasible'] else None
        self._Tf_ref   = res['Tf'] if (res is not None and res['feasible']) else self.max_t

        return self._get_obs(), {}

    # ── Step ──────────────────────────────────────────────────────────────
    def step(self, action):
        ctrl = self._denorm_action(np.clip(action, -1, 1))

        # Integrate one step with RK4
        def rhs(t, x):
            return f18_6dof_numpy(t, x, ctrl)

        x0 = self.x.copy()
        k1 = rhs(self.t, x0)
        k2 = rhs(self.t + self.dt/2, x0 + self.dt/2*k1)
        k3 = rhs(self.t + self.dt/2, x0 + self.dt/2*k2)
        k4 = rhs(self.t + self.dt,   x0 + self.dt*k3)
        x_new = x0 + self.dt/6*(k1 + 2*k2 + 2*k3 + k4)

        # Quaternion normalisation
        qn = np.linalg.norm(x_new[6:10])
        if qn > 1e-6: x_new[6:10] /= qn
        self.x = x_new
        self.t += self.dt

        # ── Reward ────────────────────────────────────────────────────────
        reward = self._compute_reward(ctrl)

        # ── Terminal conditions ────────────────────────────────────────────
        terminated = self._check_terminal()
        truncated  = (self.t >= self.max_t)

        if terminated:
            reward += self.r_terminal

        info = {'t': self.t, 'pair': (self.i_src, self.j_tgt)}
        return self._get_obs(), reward, terminated, truncated, info

    def _compute_reward(self, ctrl):
        #\"\"\"
        #Shaped reward for aggressive evasion:
        #  - Goal proximity (dense shaping in velocity + attitude space)
        #  - Body rate magnitude (aggressiveness)
         # - Time step penalty
          #- Control effort penalty
        #\"\"\"
        x, xg = self.x, self.x_goal

        # Velocity error
        dv = x[:3] - xg[:3]
        r_vel = -self.w_goal * np.sum(dv**2) / (300.**2)

        # Attitude error (quaternion geodesic distance)
        q_err = 1.0 - abs(np.dot(x[6:10], xg[6:10]))
        r_att = -self.w_goal * 5.0 * q_err

        # Aggressiveness: reward high body rates
        r_agg = self.w_agg * (abs(x[3]) + abs(x[4]) + abs(x[5]))

        # Time penalty
        r_time = -self.w_time

        # Control effort (normalised)
        ctrl_n = ctrl / CTRL_SCALE
        r_ctrl = -self.w_ctrl * np.sum(ctrl_n**2)

        return float(r_vel + r_att + r_agg + r_time + r_ctrl)

    def _check_terminal(self):
        #\"\"\"Terminal if velocity + quaternion match goal within tolerance.#\"\"\"
        x, xg = self.x, self.x_goal
        dv    = np.linalg.norm(x[:3] - xg[:3])
        q_err = 1.0 - abs(np.dot(x[6:10], xg[6:10]))
        return (dv < 3.0) and (q_err < 0.05)   # 3 m/s vel, ~10° attitude


print("✓ F18EvasionEnv defined")
print("  obs_dim=26 (state 13 + goal 13), action_dim=4")
print("  Reward: goal-tracking + aggressiveness - time - ctrl_effort + terminal bonus")

# Quick sanity check
env_test = F18EvasionEnv(TRIM_LIBRARY, TRANSITION_LIBRARY, CANDIDATES)
obs, _ = env_test.reset()
obs2, rew, term, trunc, info = env_test.step(env_test.action_space.sample())
print(f"  Sanity: obs shape={obs.shape}, reward={rew:.4f}, terminated={term}")


✓ F18EvasionEnv defined
  obs_dim=26 (state 13 + goal 13), action_dim=4
  Reward: goal-tracking + aggressiveness - time - ctrl_effort + terminal bonus
  Sanity: obs shape=(26,), reward=0.6131, terminated=False


In [12]:

# ═══════════════════════════════════════════════════════════════════════════
#  SAC Implementation — PyTorch
# ═══════════════════════════════════════════════════════════════════════════

LOG_SIG_MAX =  2.0
LOG_SIG_MIN = -20.0
EPSILON     =  1e-6

def mlp(in_dim, hidden, out_dim, activation=nn.ReLU):
    layers = []
    dims   = [in_dim] + hidden
    for i in range(len(dims)-1):
        layers += [nn.Linear(dims[i], dims[i+1]), activation()]
    layers.append(nn.Linear(dims[-1], out_dim))
    return nn.Sequential(*layers)


class GaussianActor(nn.Module):
    #"\"\"
    #Squashed Gaussian policy.
    #Input : observation (obs_dim,)
    #Output: action ∈ [-1,1]^{act_dim}  via tanh squashing
    #"\"\"
    def __init__(self, obs_dim, act_dim, hidden=(256,256)):
        super().__init__()
        self.net     = mlp(obs_dim, list(hidden), hidden[-1])
        self.mu_head = nn.Linear(hidden[-1], act_dim)
        self.ls_head = nn.Linear(hidden[-1], act_dim)

    def forward(self, obs):
        h        = F.relu(self.net(obs))
        mu       = self.mu_head(h)
        log_sig  = self.ls_head(h).clamp(LOG_SIG_MIN, LOG_SIG_MAX)
        return mu, log_sig

    def sample(self, obs):
        mu, log_sig = self(obs)
        sig = log_sig.exp()
        dist= torch.distributions.Normal(mu, sig)
        x_t = dist.rsample()                      # reparameterisation trick
        a   = torch.tanh(x_t)
        # Log-prob with tanh correction
        log_p = dist.log_prob(x_t) - torch.log(1 - a**2 + EPSILON)
        log_p = log_p.sum(dim=-1, keepdim=True)
        return a, log_p, torch.tanh(mu)


class QNetwork(nn.Module):
    #"\"\"Double Q-network.\"\"\"
    def __init__(self, obs_dim, act_dim, hidden=(256,256)):
        super().__init__()
        self.q1 = mlp(obs_dim+act_dim, list(hidden), 1)
        self.q2 = mlp(obs_dim+act_dim, list(hidden), 1)

    def forward(self, obs, act):
        x = torch.cat([obs, act], dim=-1)
        return self.q1(x), self.q2(x)


# ── Replay Buffer ──────────────────────────────────────────────────────────
Transition = tuple   # (obs, act, rew, obs2, done)

class ReplayBuffer:
    def __init__(self, capacity=int(1e6)):
        self.buf = deque(maxlen=capacity)

    def push(self, *args):
        self.buf.append(tuple(a for a in args))

    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        obs, act, rew, obs2, done = zip(*batch)
        to_t = lambda x: torch.FloatTensor(np.array(x)).to(DEVICE)
        return to_t(obs), to_t(act), to_t(rew).unsqueeze(1), \
               to_t(obs2), to_t(done).unsqueeze(1)

    def __len__(self): return len(self.buf)


# ── SAC Agent ─────────────────────────────────────────────────────────────
class SACAgent:
    def __init__(self, obs_dim, act_dim,
                 hidden=(256,256),
                 lr=3e-4,
                 gamma=0.99,
                 tau=0.005,
                 alpha_init=0.2,
                 auto_entropy=True,
                 target_entropy=None,
                 batch_size=256,
                 buffer_size=int(1e6)):

        self.gamma      = gamma
        self.tau        = tau
        self.alpha      = alpha_init
        self.auto_ent   = auto_entropy
        self.batch_size = batch_size

        # Networks
        self.actor   = GaussianActor(obs_dim, act_dim, hidden).to(DEVICE)
        self.critic  = QNetwork(obs_dim, act_dim, hidden).to(DEVICE)
        self.critic_t= copy.deepcopy(self.critic).to(DEVICE)
        for p in self.critic_t.parameters(): p.requires_grad_(False)

        # Optimisers
        self.opt_a = optim.Adam(self.actor.parameters(),  lr=lr)
        self.opt_c = optim.Adam(self.critic.parameters(), lr=lr)

        # Automatic entropy
        if auto_entropy:
            self.target_ent = target_entropy if target_entropy else -act_dim
            self.log_alpha  = torch.zeros(1, requires_grad=True, device=DEVICE)
            self.opt_alpha  = optim.Adam([self.log_alpha], lr=lr)

        self.buffer = ReplayBuffer(buffer_size)
        self.updates = 0

    @property
    def alpha_val(self):
        return self.log_alpha.exp().item() if self.auto_ent else self.alpha

    def select_action(self, obs, deterministic=False):
        obs_t = torch.FloatTensor(obs).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            if deterministic:
                _, _, a = self.actor.sample(obs_t)
            else:
                a, _, _ = self.actor.sample(obs_t)
        return a.cpu().numpy()[0]

    def update(self):
        if len(self.buffer) < self.batch_size:
            return {}

        obs, act, rew, obs2, done = self.buffer.sample(self.batch_size)

        with torch.no_grad():
            a2, log_p2, _ = self.actor.sample(obs2)
            q1_t, q2_t    = self.critic_t(obs2, a2)
            q_min_t       = torch.min(q1_t, q2_t) - self.alpha_val * log_p2
            y             = rew + self.gamma * (1 - done) * q_min_t

        # ── Critic update ──────────────────────────────────────────────
        q1, q2   = self.critic(obs, act)
        loss_c   = F.mse_loss(q1, y) + F.mse_loss(q2, y)
        self.opt_c.zero_grad(); loss_c.backward(); self.opt_c.step()

        # ── Actor update ───────────────────────────────────────────────
        a_new, log_p_new, _ = self.actor.sample(obs)
        q1_new, q2_new      = self.critic(obs, a_new)
        q_new               = torch.min(q1_new, q2_new)
        loss_a              = (self.alpha_val * log_p_new - q_new).mean()
        self.opt_a.zero_grad(); loss_a.backward(); self.opt_a.step()

        # ── Entropy coefficient update ─────────────────────────────────
        loss_e = None
        if self.auto_ent:
            loss_e = -(self.log_alpha * (log_p_new + self.target_ent).detach()).mean()
            self.opt_alpha.zero_grad(); loss_e.backward(); self.opt_alpha.step()

        # ── Soft target update ─────────────────────────────────────────
        for p, pt in zip(self.critic.parameters(), self.critic_t.parameters()):
            pt.data.copy_(self.tau*p.data + (1-self.tau)*pt.data)

        self.updates += 1
        return dict(loss_c=loss_c.item(), loss_a=loss_a.item(),
                    alpha=self.alpha_val,
                    loss_e=loss_e.item() if loss_e is not None else 0.)


print("✓ SAC agent defined")
print("  Actor : GaussianActor(256,256)")
print("  Critic: Double-Q QNetwork(256,256)")
print("  Auto-entropy tuning: ON")


✓ SAC agent defined
  Actor : GaussianActor(256,256)
  Critic: Double-Q QNetwork(256,256)
  Auto-entropy tuning: ON


In [13]:

def normalise_control(ctrl):
    #"\"\"Map physical control → [-1,1] (inverse of env._denorm_action).\"\"\"
    T, de, da, dr = ctrl
    a0 = 2*(T  - A.thrust_min)  / (A.thrust_max - A.thrust_min) - 1
    a1 = de / A.delta_e_max
    a2 = da / A.delta_a_max
    a3 = dr / A.delta_r_max
    return np.clip([a0, a1, a2, a3], -1, 1)


def build_bc_dataset(trans_lib, trim_lib, candidates):
    #"\"\"
    #Build (observation, action) pairs from OCP library.
    #bs = [state_norm | goal_norm]  (matches env observation)
    #act = normalised control [-1,1]^4
    #"\"\"
    obs_list, act_list = [], []
    for i, j in candidates:
        res = trans_lib[i][j]
        if not res['feasible']: continue
        X, U = res['X'], res['U']
        xg   = trim_lib[j]['state']
        N_k  = len(U)
        for k in range(N_k):
            xk   = X[k]
            obs  = np.concatenate([
                np.clip(xk / STATE_SCALE, -5., 5.),
                np.clip(xg / STATE_SCALE, -5., 5.)
            ]).astype(np.float32)
            act  = normalise_control(U[k]).astype(np.float32)
            obs_list.append(obs); act_list.append(act)

    return np.array(obs_list), np.array(act_list)


def behaviour_clone(agent, obs_arr, act_arr,
                    n_epochs=30, batch_size=512, lr=3e-4):
    #"\"\"Supervised pre-training of actor network.\"\"\"
    if len(obs_arr) == 0:
        print("  No BC data available — skipping pre-training."); return []

    opt   = optim.Adam(agent.actor.parameters(), lr=lr)
    n     = len(obs_arr)
    losses= []
    for epoch in range(n_epochs):
        idx   = np.random.permutation(n)
        epoch_loss = 0.0; batches = 0
        for start in range(0, n, batch_size):
            b    = idx[start:start+batch_size]
            o_t  = torch.FloatTensor(obs_arr[b]).to(DEVICE)
            a_t  = torch.FloatTensor(act_arr[b]).to(DEVICE)
            # Actor deterministic output (mean)
            mu, ls = agent.actor(o_t)
            a_pred = torch.tanh(mu)
            loss   = F.mse_loss(a_pred, a_t)
            opt.zero_grad(); loss.backward(); opt.step()
            epoch_loss += loss.item(); batches += 1
        avg = epoch_loss/batches
        losses.append(avg)
        if (epoch+1) % 5 == 0:
            print(f"  BC Epoch {epoch+1:3d}/{n_epochs}  loss={avg:.5f}")
    return losses


# ── Build BC dataset ──────────────────────────────────────────────────────
print("Building behaviour cloning dataset from OCP library ...")
BC_OBS, BC_ACT = build_bc_dataset(TRANSITION_LIBRARY, TRIM_LIBRARY, CANDIDATES)
print(f"  BC dataset: {len(BC_OBS)} (obs, act) pairs from {sum(TRANSITION_LIBRARY[i][j]['feasible'] for i,j in CANDIDATES)} feasible trajectories")


Building behaviour cloning dataset from OCP library ...
  BC dataset: 320 (obs, act) pairs from 4 feasible trajectories


In [14]:

# ═══════════════════════════════════════════════════════════════════════════
#  Full Training Configuration
# ═══════════════════════════════════════════════════════════════════════════

CFG = dict(
    obs_dim         = 26,
    act_dim         = 4,
    hidden          = (256, 256),
    lr              = 3e-4,
    gamma           = 0.99,
    tau             = 0.005,
    buffer_size     = int(5e5),
    batch_size      = 256,
    start_steps     = 2_000,      # random exploration before training
    update_every    = 1,
    updates_per_step= 1,
    total_steps     = 10_000,    # increase to 1M for full training
    eval_every      = 2_000,
    eval_episodes   = 20,
    bc_epochs       = 30,
    save_file       = 'sac_f18_evasion.pt',
    log_every       = 2_000,
)

FORCE_RETRAIN = False
MODEL_FILE = CFG['save_file']


def evaluate_agent(agent, env, n_eps=20, deterministic=True):
    #\"\"\"Evaluate mean return and success rate.\"\"\"
    returns, successes = [], []
    for _ in range(n_eps):
        obs, _ = env.reset()
        done = False; ep_ret = 0.
        while not done:
            act  = agent.select_action(obs, deterministic=deterministic)
            obs, rew, term, trunc, _ = env.step(act)
            ep_ret += rew
            done = term or trunc
        returns.append(ep_ret)
        successes.append(float(term))   # only terminal (not truncated) = success
    return np.mean(returns), np.std(returns), np.mean(successes)


def train_sac(cfg, env, agent, bc_obs, bc_act):
    #\"\"\"
    #Main SAC training loop.
    #Phase 1: Behaviour Cloning pre-training
    #Phase 2: RL fine-tuning with experience replay
    #\"\"\"
    # ── Phase 1: BC pre-training ──────────────────────────────────────────
    print("=" * 60)
    print("PHASE 1: Behaviour Cloning pre-training")
    print("=" * 60)
    bc_losses = behaviour_clone(agent, bc_obs, bc_act,
                                n_epochs=cfg['bc_epochs'],
                                batch_size=cfg['batch_size'])

    # Also fill replay buffer with BC data
    print(f"  Seeding replay buffer with {min(len(bc_obs), cfg['buffer_size']//4)} BC samples ...")
    idx = np.random.permutation(len(bc_obs))[:cfg['buffer_size']//4]
    for k in idx:
        # Fake next-obs and done for BC buffer entries (used just to warm buffer)
        agent.buffer.push(bc_obs[k], bc_act[k], 0.0, bc_obs[k], 1.0)

    # ── Phase 2: RL training ──────────────────────────────────────────────
    print("=" * 60)
    print("PHASE 2: SAC reinforcement learning")
    print("=" * 60)

    obs, _     = env.reset()
    ep_ret     = 0.0
    ep_steps   = 0
    ep_num     = 0

    metrics = dict(returns=[], successes=[], loss_c=[], loss_a=[], alphas=[])
    eval_returns = []

    t_start = time.time()

    for step in range(1, cfg['total_steps']+1):
        # ── Collect experience ─────────────────────────────────────────
        if step < cfg['start_steps']:
            act = env.action_space.sample()
        else:
            act = agent.select_action(obs, deterministic=False)

        obs2, rew, term, trunc, info = env.step(act)
        done = term or trunc

        # Store (mask terminal due to timeout — don't mask true terminal)
        agent.buffer.push(obs, act, rew, obs2, float(term))

        obs     = obs2
        ep_ret += rew
        ep_steps+= 1

        if done:
            metrics['returns'].append(ep_ret)
            metrics['successes'].append(float(term))
            obs, _ = env.reset()
            ep_ret = 0.; ep_steps = 0; ep_num += 1

        # ── Update ────────────────────────────────────────────────────
        if step >= cfg['start_steps'] and step % cfg['update_every'] == 0:
            for _ in range(cfg['updates_per_step']):
                info_u = agent.update()
                if info_u:
                    metrics['loss_c'].append(info_u['loss_c'])
                    metrics['loss_a'].append(info_u['loss_a'])
                    metrics['alphas'].append(info_u['alpha'])

        # ── Logging ───────────────────────────────────────────────────
        if step % cfg['log_every'] == 0:
            mean_ret  = np.mean(metrics['returns'][-50:]) if metrics['returns'] else float('nan')
            mean_suc  = np.mean(metrics['successes'][-50:]) if metrics['successes'] else float('nan')
            mean_lc   = np.mean(metrics['loss_c'][-100:]) if metrics['loss_c'] else float('nan')
            mean_la   = np.mean(metrics['loss_a'][-100:]) if metrics['loss_a'] else float('nan')
            alpha_v   = np.mean(metrics['alphas'][-100:]) if metrics['alphas'] else float('nan')
            elapsed   = (time.time()-t_start)/60
            print(f"Step {step:>7d} | Ep{ep_num:>5d} | "
                  f"RetMean={mean_ret:>8.1f} | Succ={mean_suc:.2f} | "
                  f"Lc={mean_lc:.4f} | La={mean_la:.4f} | α={alpha_v:.3f} | "
                  f"{elapsed:.1f}m")

        # ── Evaluation ────────────────────────────────────────────────
        if step % cfg['eval_every'] == 0:
            m, s, succ = evaluate_agent(agent, env, cfg['eval_episodes'])
            eval_returns.append((step, m, s, succ))
            print(f"  ► EVAL  step={step}  mean={m:.1f}±{s:.1f}  success={succ:.2f}")
            # Save checkpoint
            torch.save({'actor': agent.actor.state_dict(),
                        'critic': agent.critic.state_dict(),
                        'step': step}, MODEL_FILE)

    return metrics, eval_returns, bc_losses


# ── Instantiate agent & environment ──────────────────────────────────────────
ENV  = F18EvasionEnv(TRIM_LIBRARY, TRANSITION_LIBRARY, CANDIDATES,
                     dt=0.05, max_t=30.0)
AGENT = SACAgent(obs_dim=CFG['obs_dim'], act_dim=CFG['act_dim'],
                 hidden=CFG['hidden'],   lr=CFG['lr'],
                 gamma=CFG['gamma'],     tau=CFG['tau'],
                 buffer_size=CFG['buffer_size'],
                 batch_size=CFG['batch_size'])

if os.path.exists(MODEL_FILE) and not FORCE_RETRAIN:
    ckpt = torch.load(MODEL_FILE, map_location=DEVICE)
    AGENT.actor.load_state_dict(ckpt['actor'])
    AGENT.critic.load_state_dict(ckpt['critic'])
    print(f"✓ Loaded checkpoint from '{MODEL_FILE}' (step {ckpt.get('step','?')})")
    METRICS, EVAL_RETS, BC_LOSSES = {}, [], []
else:
    print(f"Starting fresh training — {CFG['total_steps']:,} steps")
    METRICS, EVAL_RETS, BC_LOSSES = train_sac(CFG, ENV, AGENT, BC_OBS, BC_ACT)
    print("\\n✓ Training complete")


Starting fresh training — 10,000 steps
PHASE 1: Behaviour Cloning pre-training
  BC Epoch   5/30  loss=0.09910
  BC Epoch  10/30  loss=0.03515
  BC Epoch  15/30  loss=0.02159
  BC Epoch  20/30  loss=0.02066
  BC Epoch  25/30  loss=0.01947
  BC Epoch  30/30  loss=0.02048
  Seeding replay buffer with 320 BC samples ...
PHASE 2: SAC reinforcement learning
Step    2000 | Ep    4 | RetMean=  -416.3 | Succ=0.25 | Lc=323.2192 | La=-1.4044 | α=1.000 | 0.1m
  ► EVAL  step=2000  mean=-1087.7±412.0  success=0.05
Step    4000 | Ep    8 | RetMean=   534.8 | Succ=0.12 | Lc=17.6184 | La=-32.5403 | α=0.590 | 1.6m
  ► EVAL  step=4000  mean=2822.1±524.7  success=0.00
Step    6000 | Ep   12 | RetMean=   990.0 | Succ=0.08 | Lc=12.9422 | La=-68.1470 | α=0.354 | 3.1m
  ► EVAL  step=6000  mean=1489.7±1756.8  success=0.00
Step    8000 | Ep   16 | RetMean=  1104.0 | Succ=0.06 | Lc=12.0016 | La=-104.6012 | α=0.231 | 4.6m
  ► EVAL  step=8000  mean=1991.3±2900.2  success=0.05
Step   10000 | Ep   20 | RetMean=  13

In [15]:

def plot_training_curves(metrics, eval_rets, bc_losses):
    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    fig.suptitle('SAC Training Diagnostics — F-18 Evasive Path Planning',
                 fontsize=13, fontweight='bold')

    # BC loss
    ax = axes[0,0]
    if bc_losses:
        ax.plot(bc_losses, 'b-', lw=2)
        ax.set_title('Behaviour Cloning Loss'); ax.set_xlabel('Epoch')
        ax.set_ylabel('MSE'); ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No BC data', ha='center', va='center',
                transform=ax.transAxes)

    # Episode returns
    ax = axes[0,1]
    if metrics.get('returns'):
        w = min(50, len(metrics['returns']))
        smooth = np.convolve(metrics['returns'],
                             np.ones(w)/w, mode='valid')
        ax.plot(metrics['returns'], alpha=0.2, color='steelblue')
        ax.plot(range(w-1, len(metrics['returns'])), smooth, 'b-', lw=2)
        ax.set_title('Episode Returns'); ax.set_xlabel('Episode')
        ax.set_ylabel('Return'); ax.grid(True, alpha=0.3)

    # Success rate
    ax = axes[0,2]
    if metrics.get('successes'):
        w = min(100, len(metrics['successes']))
        smooth = np.convolve(metrics['successes'], np.ones(w)/w, mode='valid')
        ax.plot(range(w-1, len(metrics['successes'])), smooth, 'g-', lw=2)
        ax.set_title('Success Rate (rolling)'); ax.set_xlabel('Episode')
        ax.set_ylabel('Rate'); ax.set_ylim(0,1); ax.grid(True, alpha=0.3)

    # Critic loss
    ax = axes[1,0]
    if metrics.get('loss_c'):
        w = min(100, len(metrics['loss_c']))
        smooth = np.convolve(metrics['loss_c'], np.ones(w)/w, mode='valid')
        ax.plot(range(w-1, len(metrics['loss_c'])), smooth, 'r-', lw=2)
        ax.set_title('Critic Loss'); ax.set_xlabel('Update step')
        ax.set_ylabel('MSE'); ax.grid(True, alpha=0.3)

    # Actor loss
    ax = axes[1,1]
    if metrics.get('loss_a'):
        w = min(100, len(metrics['loss_a']))
        smooth = np.convolve(metrics['loss_a'], np.ones(w)/w, mode='valid')
        ax.plot(range(w-1, len(metrics['loss_a'])), smooth, 'm-', lw=2)
        ax.set_title('Actor Loss'); ax.set_xlabel('Update step')
        ax.set_ylabel('Loss'); ax.grid(True, alpha=0.3)

    # Entropy / alpha
    ax = axes[1,2]
    if metrics.get('alphas'):
        ax.plot(metrics['alphas'], 'k-', alpha=0.3, lw=1)
        ax.set_title('Entropy coefficient α'); ax.set_xlabel('Update step')
        ax.set_ylabel('α'); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=120, bbox_inches='tight')
    plt.close()
    print("✓ Saved: training_curves.png")

    # Eval curve
    if eval_rets:
        fig, ax = plt.subplots(figsize=(10, 4))
        steps = [e[0] for e in eval_rets]
        means = [e[1] for e in eval_rets]
        stds  = [e[2] for e in eval_rets]
        succs = [e[3] for e in eval_rets]
        ax.fill_between(steps, np.array(means)-np.array(stds),
                        np.array(means)+np.array(stds), alpha=0.2)
        ax.plot(steps, means, 'b-o', lw=2, label='Mean return')
        ax2 = ax.twinx()
        ax2.plot(steps, succs, 'g--s', lw=2, label='Success rate')
        ax.set_xlabel('Training step'); ax.set_ylabel('Return', color='b')
        ax2.set_ylabel('Success rate', color='g'); ax2.set_ylim(0,1)
        ax.set_title('Evaluation Curves')
        ax.grid(True, alpha=0.3)
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1+lines2, labels1+labels2, loc='lower right')
        plt.tight_layout()
        plt.savefig('eval_curves.png', dpi=120, bbox_inches='tight')
        plt.close()
        print("✓ Saved: eval_curves.png")


plot_training_curves(METRICS, EVAL_RETS, BC_LOSSES)


✓ Saved: training_curves.png
✓ Saved: eval_curves.png


In [16]:

# ═══════════════════════════════════════════════════════════════════════════
#  Roll out the SAC policy and compare vs OCP reference
# ═══════════════════════════════════════════════════════════════════════════

def rollout_policy(agent, env, i_src, j_tgt, deterministic=True, seed=0):
    #\"\"\"
    #un a full episode for a specific (i,j) pair.
    #Returns trajectory arrays.
    #\"\"\"
    obs, _ = env.reset(seed=seed)
    # Force the desired pair by resetting manually
    env.i_src  = i_src
    env.j_tgt  = j_tgt
    pi = env.trim_lib[i_src]
    pj = env.trim_lib[j_tgt]
    env.x      = pi['state'].copy()
    env.x_goal = pj['state'].copy()
    env.t      = 0.0
    res = env.trans_lib[i_src][j_tgt]
    env._ref_traj = res if res['feasible'] else None
    obs = env._get_obs()

    states = [env.x.copy()]
    ctrls  = []
    rews   = []
    done   = False

    while not done:
        act = agent.select_action(obs, deterministic=deterministic)
        obs, rew, term, trunc, info = env.step(act)
        states.append(env.x.copy())
        ctrls.append(env._denorm_action(act))
        rews.append(rew)
        done = term or trunc

    return np.array(states), np.array(ctrls), np.array(rews)


def compare_ocp_vs_rl(trans_lib, trim_lib, agent, env, i, j):
    #\"\"\"Side-by-side comparison of OCP and RL trajectories.\"\"\"
    ref = trans_lib[i][j]
    rl_states, rl_ctrl, rl_rews = rollout_policy(agent, env, i, j)

    pi_lbl = trim_lib[i]['label']
    pj_lbl = trim_lib[j]['label']
    title  = f"{pi_lbl} → {pj_lbl}"

    fig, axes = plt.subplots(2, 3, figsize=(18, 9))
    fig.suptitle(f'OCP vs RL Policy: {title}', fontsize=13, fontweight='bold')
    colors = {'OCP': '#1f77b4', 'RL': '#d62728'}

    # ── 3-D path ──────────────────────────────────────────────────────────
    ax = fig.add_subplot(2, 3, 1, projection='3d')
    if ref['feasible']:
        X_ref = ref['X']
        ax.plot(X_ref[:,10], X_ref[:,11], -X_ref[:,12],
                color=colors['OCP'], lw=2, label='OCP')
    ax.plot(rl_states[:,10], rl_states[:,11], -rl_states[:,12],
            color=colors['RL'], lw=2, ls='--', label='RL')
    ax.scatter(*trim_lib[i]['state'][10:13]*[1,1,-1], c='g', s=60, zorder=5)
    ax.scatter(*trim_lib[j]['state'][10:13]*[1,1,-1], c='r', s=60, zorder=5)
    ax.set_xlabel('N'); ax.set_ylabel('E'); ax.set_zlabel('Alt')
    ax.set_title('3-D Trajectory'); ax.legend(fontsize=8)

    # Helper: RL time axis
    t_rl = np.arange(len(rl_states)) * env.dt

    # ── Airspeed ──────────────────────────────────────────────────────────
    ax = axes[0,1]
    if ref['feasible']:
        V_ref = np.sqrt(np.sum(ref['X'][:,:3]**2, axis=1))
        ax.plot(ref['T'], V_ref, color=colors['OCP'], lw=2, label='OCP')
    V_rl = np.sqrt(np.sum(rl_states[:,:3]**2, axis=1))
    ax.plot(t_rl, V_rl, color=colors['RL'], lw=2, ls='--', label='RL')
    ax.set_title('Airspeed'); ax.set_xlabel('t [s]'); ax.set_ylabel('V [m/s]')
    ax.legend(); ax.grid(True, alpha=0.3)

    # ── Body rates ────────────────────────────────────────────────────────
    ax = axes[0,2]
    for k, (col, ls) in enumerate([('r','p'), ('g','q'), ('b','r')]):
        if ref['feasible']:
            ax.plot(ref['T'], np.degrees(ref['X'][:,3+k]),
                    color=col, lw=2, alpha=0.4, label=f'OCP-{ls}')
        ax.plot(t_rl, np.degrees(rl_states[:,3+k]),
                color=col, lw=2, ls='--', label=f'RL-{ls}')
    ax.set_title('Body Rates'); ax.set_xlabel('t [s]')
    ax.set_ylabel('[deg/s]'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    # ── Thrust ───────────────────────────────────────────────────────────
    ax = axes[1,0]
    t_ctrl = np.arange(len(rl_ctrl)) * env.dt
    if ref['feasible']:
        ax.plot(ref['T'][:-1], ref['U'][:,0]/1e3,
                color=colors['OCP'], lw=2, label='OCP')
    ax.plot(t_ctrl, rl_ctrl[:,0]/1e3, color=colors['RL'], lw=2, ls='--', label='RL')
    ax.set_title('Thrust'); ax.set_xlabel('t [s]'); ax.set_ylabel('[kN]')
    ax.legend(); ax.grid(True, alpha=0.3)

    # ── Control surfaces ──────────────────────────────────────────────────
    ax = axes[1,1]
    for k, (col, lbl) in enumerate([('r','δe'),('g','δa'),('b','δr')]):
        if ref['feasible']:
            ax.plot(ref['T'][:-1], np.degrees(ref['U'][:,k+1]),
                    color=col, lw=2, alpha=0.4, label=f'OCP-{lbl}')
        ax.plot(t_ctrl, np.degrees(rl_ctrl[:,k+1]),
                color=col, lw=2, ls='--', label=f'RL-{lbl}')
    ax.set_title('Control Surfaces'); ax.set_xlabel('t [s]')
    ax.set_ylabel('[deg]'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    # ── Episode reward ─────────────────────────────────────────────────────
    ax = axes[1,2]
    ax.plot(t_ctrl, rl_rews, color='purple', lw=2)
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.fill_between(t_ctrl, rl_rews, 0,
                    where=np.array(rl_rews)>0, alpha=0.3, color='g')
    ax.fill_between(t_ctrl, rl_rews, 0,
                    where=np.array(rl_rews)<0, alpha=0.3, color='r')
    ax.set_title('Step Rewards'); ax.set_xlabel('t [s]')
    ax.set_ylabel('r'); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fname = f'compare_{pi_lbl}_{pj_lbl}.png'
    plt.savefig(fname, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {fname}  |  RL total return={sum(rl_rews):.1f}"
          f"  |  duration={t_rl[-1]:.1f}s")


# ── Run comparisons for first 3 feasible pairs ────────────────────────────────
print("OCP vs RL trajectory comparisons:")
count = 0
for i, j in CANDIDATES:
    if TRANSITION_LIBRARY[i][j]['feasible']:
        compare_ocp_vs_rl(TRANSITION_LIBRARY, TRIM_LIBRARY, AGENT, ENV, i, j)
        count += 1
        if count >= 3: break

if count == 0:
    print("  No feasible OCP pairs found — run OCP solver first (Cell 6).")


OCP vs RL trajectory comparisons:
  Saved: compare_TRIM-000_TRIM-001.png  |  RL total return=3063.6  |  duration=30.0s
  Saved: compare_TRIM-000_TRIM-002.png  |  RL total return=3028.5  |  duration=30.0s
  Saved: compare_TRIM-000_TRIM-003.png  |  RL total return=3147.9  |  duration=30.0s


In [18]:

# ═══════════════════════════════════════════════════════════════════════════
#  Generalisation Evaluation
#  Test RL policy on EVERY feasible (i,j) pair in CANDIDATES.
#  Metrics: success rate, mean episode return, mean aggressiveness index.
# ═══════════════════════════════════════════════════════════════════════════

def aggressiveness_index(states):
    #\"\"\"
    #Quantify trajectory aggressiveness:
     # AI = mean(|p| + |q| + |r|) in deg/s + rms of body-rate change.
    #Higher = more evasive.
    #\"\"\"
    pqr = np.degrees(states[:, 3:6])
    mean_rate = np.mean(np.abs(pqr).sum(axis=1))
    dpqr_rms  = np.sqrt(np.mean(np.diff(pqr, axis=0)**2))
    return mean_rate + dpqr_rms


def generalisation_eval(agent, env, trim_lib, trans_lib, candidates,
                         deterministic=True, n_repeat=3):
    #\"\"\"
    #Evaluate SAC agent on all (i,j) pairs.
    #Returns a summary DataFrame-like dict.
    #\"\"\"
    results = []
    print(f"{'Pair':<18} {'Success':>8} {'RetMean':>10} {'AI (RL)':>10}"
          f" {'AI (OCP)':>10} {'ΔAI%':>8}")
    print("─" * 68)

    for i, j in candidates:
        pi_lbl = trim_lib[i]['label']
        pj_lbl = trim_lib[j]['label']
        pair_str = f"{pi_lbl}→{pj_lbl}"

        ep_rets = []; ep_succ = []; ai_rl_list = []
        for rep in range(n_repeat):
            states, _, rews = rollout_policy(agent, env, i, j,
                                             deterministic=deterministic,
                                             seed=rep)
            ep_rets.append(sum(rews))
            # Check terminal: final state close to goal
            xf = env.x_goal
            dv   = np.linalg.norm(states[-1,:3] - xf[:3])
            q_err= 1. - abs(np.dot(states[-1,6:10], xf[6:10]))
            ep_succ.append(float(dv < 5.0 and q_err < 0.08))
            ai_rl_list.append(aggressiveness_index(states))

        ai_rl_mean = np.mean(ai_rl_list)
        ref = trans_lib[i][j]
        ai_ocp = aggressiveness_index(ref['X']) if ref['feasible'] else float('nan')
        delta_ai = ((ai_rl_mean - ai_ocp)/max(ai_ocp,1.))*100 if np.isfinite(ai_ocp) else float('nan')

        row = dict(i=i, j=j, pair=pair_str,
                   success=np.mean(ep_succ), ret_mean=np.mean(ep_rets),
                   ai_rl=ai_rl_mean, ai_ocp=ai_ocp, delta_ai=delta_ai,
                   ocp_feasible=ref['feasible'])
        results.append(row)

        suc_str = f"{np.mean(ep_succ):.2f}"
        ai_ocp_str = f"{ai_ocp:>10.2f}" if np.isfinite(ai_ocp) else f"{'N/A':>10}"
        d_str  = f"{delta_ai:>+8.1f}%" if np.isfinite(delta_ai) else f"{'N/A':>8}"
        print(f"{pair_str:<18} {suc_str:>8} {np.mean(ep_rets):>10.1f}"
              f" {ai_rl_mean:>10.2f} {ai_ocp_str} {d_str}")

    return results


print("Running generalisation evaluation across all pairs ...\n")
GEN_RESULTS = generalisation_eval(AGENT, ENV, TRIM_LIBRARY,
                                  TRANSITION_LIBRARY, CANDIDATES,
                                  n_repeat=3)

# ── Summary statistics ────────────────────────────────────────────────────────
n_total   = len(GEN_RESULTS)
n_success = sum(r['success'] >= 0.67 for r in GEN_RESULTS)
mean_ret  = np.mean([r['ret_mean'] for r in GEN_RESULTS])
mean_ai   = np.mean([r['ai_rl']   for r in GEN_RESULTS])

feas_pairs = [r for r in GEN_RESULTS if r['ocp_feasible']]
mean_delta = (np.nanmean([r['delta_ai'] for r in feas_pairs])
              if feas_pairs else float('nan'))

print(f"\n{'═'*60}")
print(f"GENERALISATION SUMMARY")
print(f"{'─'*60}")
print(f"  Total pairs evaluated : {n_total}")
print(f"  Pairs with ≥67% success: {n_success} ({100*n_success/max(n_total,1):.1f}%)")
print(f"  Mean episode return   : {mean_ret:.1f}")
print(f"  Mean aggressiveness AI: {mean_ai:.2f} deg/s")
print(f"  Mean ΔAI vs OCP       : {mean_delta:+.1f}%  (+ = more aggressive than OCP)")
print(f"{'═'*60}")


Running generalisation evaluation across all pairs ...

Pair                Success    RetMean    AI (RL)   AI (OCP)     ΔAI%
────────────────────────────────────────────────────────────────────
TRIM-000→TRIM-001      0.00     3063.6     853.89       5.45 +15579.6%
TRIM-000→TRIM-002      0.00     3028.5     844.71       2.20 +38329.9%
TRIM-000→TRIM-003      0.00     3147.9     867.32       3.41 +25363.4%
TRIM-000→TRIM-004      0.00     3135.8     863.64        N/A      N/A
TRIM-000→TRIM-005      0.00     3207.2     874.90       6.53 +13308.0%
TRIM-000→TRIM-006      0.00     3240.1     879.20        N/A      N/A
TRIM-000→TRIM-007      0.00     3062.6     842.85        N/A      N/A
TRIM-000→TRIM-015      0.00     4306.9    1055.37        N/A      N/A
TRIM-000→TRIM-016      0.00     4282.8    1044.73        N/A      N/A
TRIM-000→TRIM-020      0.00     4119.9    1144.88        N/A      N/A
TRIM-000→TRIM-021      0.00     4199.1    1155.43        N/A      N/A
TRIM-000→TRIM-025      0.00    

KeyboardInterrupt: 

In [ ]:

def plot_generalisation_heatmap(results, trim_lib, candidates, metric='success'):
    #\"\"\"Visualise success rate / aggressiveness over all (i,j) pairs.\"\"\"
    N_p = len(trim_lib)
    labels = [p['label'] for p in trim_lib]

    mat = np.full((N_p, N_p), np.nan)
    for r in results:
        mat[r['i'], r['j']] = r[metric]

    fig, ax = plt.subplots(figsize=(11, 9))
    im = ax.imshow(mat, cmap='RdYlGn' if metric=='success' else 'plasma',
                   vmin=0 if metric=='success' else None, vmax=1 if metric=='success' else None)
    ax.set_xticks(range(N_p)); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(N_p)); ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('Goal state j');  ax.set_ylabel('Initial state i')
    title = {'success': 'Success Rate', 'ai_rl': 'Aggressiveness Index (RL)',
             'ret_mean': 'Mean Episode Return'}.get(metric, metric)
    ax.set_title(f'Generalisation Heatmap — {title}', fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8)

    # Annotate cells
    for r in results:
        val = r[metric]
        if np.isfinite(val):
            ax.text(r['j'], r['i'], f'{val:.2f}', ha='center', va='center',
                    fontsize=6, color='black')

    plt.tight_layout()
    fname = f'heatmap_{metric}.png'
    plt.savefig(fname, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {fname}")


for m in ['success', 'ai_rl', 'ret_mean']:
    plot_generalisation_heatmap(GEN_RESULTS, TRIM_LIBRARY, CANDIDATES, metric=m)


In [ ]:

# ═══════════════════════════════════════════════════════════════════════════
#  Online Fine-Tuning
#  Given any new (s0, sf) outside the training library,
#  quickly adapt the pre-trained SAC agent with a short online phase.
# ═══════════════════════════════════════════════════════════════════════════

def online_finetune(agent, env, x0_np, xf_np,
                    finetune_steps=5000, dt=0.05, max_t=30.):
    #\"\"\"
    #Fine-tune a pre-trained SAC agent on a specific (x0, xf) pair.
    #Creates a single-pair sub-environment and runs SAC updates.
    #\"\"\"
    # Wrap in a minimal env with fixed pair
    class FixedPairEnv(F18EvasionEnv):
        def reset(self, seed=None, options=None):
            super(gym.Env, self).__init__()
            self.x      = x0_np.copy()
            self.x_goal = xf_np.copy()
            self.t      = 0.
            self._ref_traj = None; self._Tf_ref = max_t
            q = self.x[6:10]; self.x[6:10] /= np.linalg.norm(q)
            return self._get_obs(), {}

    sub_env = FixedPairEnv(env.trim_lib, env.trans_lib, env.candidates,
                            dt=dt, max_t=max_t)

    # Copy agent weights (shallow fine-tune — don't corrupt shared policy)
    ft_agent = SACAgent(obs_dim=26, act_dim=4, hidden=(256,256),
                        lr=1e-4, batch_size=128, buffer_size=50_000)
    ft_agent.actor.load_state_dict(copy.deepcopy(agent.actor.state_dict()))
    ft_agent.critic.load_state_dict(copy.deepcopy(agent.critic.state_dict()))
    ft_agent.critic_t.load_state_dict(copy.deepcopy(agent.critic_t.state_dict()))

    obs, _ = sub_env.reset()
    ep_rets = []; best_ret = -np.inf
    ep_r = 0.

    for step in range(finetune_steps):
        act = ft_agent.select_action(obs, deterministic=False)
        obs2, rew, term, trunc, _ = sub_env.step(act)
        ft_agent.buffer.push(obs, act, rew, obs2, float(term))
        obs = obs2; ep_r += rew
        if term or trunc:
            ep_rets.append(ep_r)
            best_ret = max(best_ret, ep_r)
            obs, _ = sub_env.reset()
            ep_r = 0.
        if step > 512:
            ft_agent.update()

    # Final rollout with fine-tuned policy
    states, ctrls, rews = rollout_policy(ft_agent, sub_env, 0, 0,
                                          deterministic=True, seed=99)
    print(f"  Fine-tune done: {finetune_steps} steps | "
          f"best_ret={best_ret:.1f} | final_dur={len(states)*dt:.1f}s")
    return ft_agent, states, ctrls, rews


# ── Example: arbitrary new pair (interpolated between two trim states) ────────
print("Online fine-tuning demo on a new (s0, sf) pair ...")

pi_new = TRIM_LIBRARY[0]
pj_new = TRIM_LIBRARY[-1]
x0_new = pi_new['state'].copy()
xf_new = pj_new['state'].copy()
# Add 10% perturbation to make it truly "new"
x0_new[:3] *= 1.08
x0_new[6:10] /= np.linalg.norm(x0_new[6:10])

ft_agent, ft_states, ft_ctrls, ft_rews = online_finetune(
    AGENT, ENV, x0_new, xf_new, finetune_steps=3000)

print(f"✓ Total fine-tune return: {sum(ft_rews):.1f}")


In [ ]:

def save_all(agent, path='f18_sac_full.pt'):
    torch.save({
        'actor_state':    agent.actor.state_dict(),
        'critic_state':   agent.critic.state_dict(),
        'critic_t_state': agent.critic_t.state_dict(),
        'log_alpha':      agent.log_alpha.item() if agent.auto_ent else agent.alpha,
        'updates':        agent.updates,
    }, path)
    print(f"✓ Model saved to '{path}'")


def load_all(path='f18_sac_full.pt'):
    ckpt = torch.load(path, map_location=DEVICE)
    agent = SACAgent(obs_dim=26, act_dim=4)
    agent.actor.load_state_dict(ckpt['actor_state'])
    agent.critic.load_state_dict(ckpt['critic_state'])
    agent.critic_t.load_state_dict(ckpt['critic_t_state'])
    if agent.auto_ent:
        agent.log_alpha.data.fill_(np.log(ckpt['log_alpha'] + 1e-8))
    agent.updates = ckpt.get('updates', 0)
    print(f"✓ Model loaded from '{path}' (updates={agent.updates})")
    return agent


save_all(AGENT, 'f18_sac_full.pt')
print("\\n─── Usage ───")
print("  agent = load_all('f18_sac_full.pt')")
print("  obs, _ = env.reset()")
print("  action = agent.select_action(obs, deterministic=True)   # inference")


